# Google Colab Notebook
Este primer notebook se corrio en google colab por la facilidad de la GPU T4 que ofrece gratis para poder correr el modelo transformer Whisper y la dialization.

## Verificar GPU
Comprueba si Colab tiene acceso a una GPU para que Whisper y la diarización corran más rápido y con mejor calidad.

In [ ]:
!nvidia-smi

Tue Sep 15 04:53:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Instalar dependencias
Instalamos las librerías necesarias para transcribir audio con Whisper y separar hablantes con diarización automática.

In [ ]:
!pip install -q faster-whisper pyannote.audio


## Configurar el token de Hugging Face

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')


## Clonar el repo con los audios

In [ ]:
!git clone https://github.com/andreschaparr0/prueba_crecere.git



fatal: destination path 'prueba_crecere' already exists and is not an empty directory.


In [ ]:
%cd prueba_crecere

/content/prueba_crecere


## Paths
Definimos las carpetas de entrada y salida para leer los audios humanos/IA y guardar cada transcripción en su carpeta correspondiente.

In [ ]:
import os

HUMANOS_DIR = "Audios/audios_humanos_censurados"
IA_DIR = "Audios/audios_ia_censurados"
OUT_DIR = "Transcripciones"

os.makedirs(f"{OUT_DIR}/humanos", exist_ok=True)
os.makedirs(f"{OUT_DIR}/ia", exist_ok=True)

print("Humanos:", len(os.listdir(HUMANOS_DIR)))
print("IA:", len(os.listdir(IA_DIR)))


Humanos: 50
IA: 50


## Cargar modelos (Whisper + diarización)
Se inicializa el modelo Whisper en modo GPU cuando está disponible y se prepara el pipeline de diarización con el token de Hugging Face. Este paso es el más pesado del proceso, pero deja listos los modelos para transcribir y etiquetar hablantes.



In [ ]:
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print("Cargando modelo Whisper...")
whisper_model = WhisperModel("large-v3", device=device, compute_type=compute_type)

print("Cargando pipeline de diarización...")
diarize_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN
)
if device == "cuda":
    diarize_pipeline.to(torch.device("cuda"))

diarize_pipeline.instantiate({
    "segmentation": {"min_duration_off": 0.0},
    "clustering": {"method": "centroid", "min_cluster_size": 5, "threshold": 0.7045654963945799}
})

print("Listo.")


Cargando modelo Whisper...
Cargando pipeline de diarización...
Listo.


## Funciones para procesar el audio (transcribirlo + dialization)

Esta celda define la lógica principal: transcribe cada audio con Whisper y luego, usando la diarización, asigna a cada tramo el hablante más probable.

process_audio toma un archivo, lo transcribe con Whisper en español y luego compara cada segmento con la diarización para asignar un hablante. Se usan filtros de VAD y umbrales para evitar ruido o texto inventado, y al final devuelve un JSON con el tiempo, número de hablantes y los segmentos resultantes.

Es la parte central del flujo: el audio entra, se divide en trozos, se etiqueta con hablante y se guarda en un formato listo para analizar

In [ ]:
def assign_speaker(seg_start, seg_end, diarization):
    """Asigna al segmento de Whisper el hablante con mayor solapamiento temporal."""
    best_speaker, best_overlap = "UNKNOWN", 0.0
    for turn, _, speaker in diarization.speaker_diarization.itertracks(yield_label=True):
        overlap = min(seg_end, turn.end) - max(seg_start, turn.start)
        if overlap > best_overlap:
            best_overlap, best_speaker = overlap, speaker
    return best_speaker

def process_audio(audio_path, file_id, source):
    segments_gen, info = whisper_model.transcribe(
        audio_path,
        language="es",
        vad_filter=True,
        vad_parameters=dict(
            threshold=0.2,              # menos agresivo que el default (0.5)
            min_silence_duration_ms=500  # solo corta silencios reales, no pausas cortas
        ),
        condition_on_previous_text=False,  # evita que un error de transcripción "contamine" los siguientes segmentos
        no_speech_threshold=0.6,           # más exigente para decidir "esto es silencio/ruido, no lo transcribas"
        compression_ratio_threshold=2.4    # descarta segmentos con texto repetitivo/sin sentido (síntoma típico de hallucinación)
    )
    diarization = diarize_pipeline(audio_path, min_speakers=2, max_speakers=2)

    segments_out = []
    for seg in segments_gen:
        speaker = assign_speaker(seg.start, seg.end, diarization)
        segments_out.append({
            "start": round(seg.start, 2),
            "end": round(seg.end, 2),
            "speaker": speaker,
            "text": seg.text.strip()
        })

    speakers_detected = len(set(s["speaker"] for s in segments_out))

    return {
        "file_id": file_id,
        "source": source,
        "duration_sec": round(info.duration, 2),
        "language": "es",
        "n_speakers_detected": speakers_detected,
        "segments": segments_out
    }



## Probarlo en un audio

In [ ]:
r_check = process_audio(f"{HUMANOS_DIR}/09115a4a-ac50-4425-9dbc-b551a0b5342c.wav", "check", "humano")
print("Hablantes:", r_check["n_speakers_detected"])
for s in r_check["segments"]:
    print(f"[{s['start']:>6.1f} - {s['end']:>6.1f}] {s['speaker']}: {s['text']}")


/usr/local/lib/python3.13/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/usr/local/lib/python3.13/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


Hablantes: 2
[   3.5 -    7.5] SPEAKER_01: Buenos días, por favor, la señora Luz Alejandra.
[   8.7 -    9.3] SPEAKER_01: Sí, con ella.
[  10.1 -   11.8] SPEAKER_01: ¿Cómo está? Nuevamente le habla.
[  13.5 -   17.1] SPEAKER_01: Por el tema de la obligación originada inicialmente con...
[  17.1 -   18.4] SPEAKER_01: ¿Cómo se encuentra el día de hoy?
[  19.1 -   20.0] SPEAKER_01: Bien, gracias.
[  21.6 -   26.9] SPEAKER_01: Le quiero contar que en este momento nos generaron la aprobación del pago total de los 10 millones de pesos.
[  27.1 -   34.9] SPEAKER_01: En este caso, pues la idea principal es poderle remitir a su merced el acuerdo de pago formal con el descuento aprobado por parte...
[  35.2 -   37.6] SPEAKER_01: usted me informó que realizaría el pago
[  37.6 -   39.9] SPEAKER_01: el día 31 de agosto
[  39.9 -   43.4] SPEAKER_01: ¿Puedo hacer un abono antes de esa fecha?
[  43.6 -   45.5] SPEAKER_01: ¿O necesariamente sería hasta ese día
[  45.5 -   47.3] SPEAKER_01: que lograrí

## Transcribir todos los audios

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT_DIR = "/content/drive/MyDrive/prueba_crecere_transcripciones"
import os
os.makedirs(f"{DRIVE_OUT_DIR}/humanos", exist_ok=True)
os.makedirs(f"{DRIVE_OUT_DIR}/ia", exist_ok=True)
print("Carpeta de salida en Drive lista.")


Mounted at /content/drive
Carpeta de salida en Drive lista.


In [ ]:
import glob, json, time

folders = [
    (HUMANOS_DIR, "humano", f"{DRIVE_OUT_DIR}/humanos"),
    (IA_DIR, "ia", f"{DRIVE_OUT_DIR}/ia"),
]

csv_rows = []
errors = []

for folder, source, out_subdir in folders:
    files = sorted(glob.glob(os.path.join(folder, "*.wav")))
    print(f"Procesando {len(files)} audios de '{source}'...")
    for i, filepath in enumerate(files):
        file_id = os.path.splitext(os.path.basename(filepath))[0]
        out_path = os.path.join(out_subdir, f"{file_id}.json")

        if os.path.exists(out_path):
            print(f"[{i+1}/{len(files)}] {file_id} ya existe, se salta")
            with open(out_path, encoding="utf-8") as f:
                r_saved = json.load(f)
            csv_rows.append({"file_id": file_id, "source": source,
                              "duration_sec": r_saved["duration_sec"],
                              "n_speakers_detected": r_saved["n_speakers_detected"],
                              "n_segments": len(r_saved["segments"])})
            continue

        try:
            t0 = time.time()
            result = process_audio(filepath, file_id, source)
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
            csv_rows.append({"file_id": file_id, "source": source,
                              "duration_sec": result["duration_sec"],
                              "n_speakers_detected": result["n_speakers_detected"],
                              "n_segments": len(result["segments"])})
            print(f"[{i+1}/{len(files)}] {file_id} OK ({time.time()-t0:.1f}s, {result['n_speakers_detected']} hablantes)")
        except Exception as e:
            print(f"[{i+1}/{len(files)}] {file_id} ERROR: {e}")
            errors.append((file_id, str(e)))

print(f"Listo. Procesados: {len(csv_rows)} | Errores: {len(errors)}")
for fid, err in errors:
    print(f" - {fid}: {err}")


Procesando 50 audios de 'humano'...
[1/50] 0445c357-e465-49ae-8067-bfa91d11b532 OK (49.9s, 2 hablantes)
[2/50] 09115a4a-ac50-4425-9dbc-b551a0b5342c OK (15.1s, 2 hablantes)
[3/50] 0bc9a430-bdcd-44a7-a92d-e71dc33c8ad5 OK (11.4s, 1 hablantes)
[4/50] 0c6ada7d-d8c5-4840-ae79-0d8d573183dc OK (24.8s, 1 hablantes)
[5/50] 0d2d37dd-38a4-40dd-ac62-b430f722ed57 OK (88.0s, 2 hablantes)
[6/50] 1c72dd55-acbb-4ebe-a4d7-446cc294a4a1 OK (31.3s, 1 hablantes)
[7/50] 1e3beb83-824b-4ad5-8483-f4ff6ea3c05e OK (22.0s, 1 hablantes)
[8/50] 1f5cda00-bbf3-4770-97dc-99760ddadc1b OK (21.7s, 2 hablantes)
[9/50] 2a92a64c-6ea8-48a0-b47a-445a3c577199 OK (103.6s, 2 hablantes)
[10/50] 2b377b6a-4fa1-4910-8d68-9ee05f7c302a OK (27.2s, 1 hablantes)
[11/50] 2d454b7b-9f12-4acb-8101-d3ba74bb84ff OK (88.8s, 2 hablantes)
[12/50] 2e50e9db-53c9-4472-ad05-1ba1032dd707 OK (16.8s, 2 hablantes)
[13/50] 306c762c-73a1-4690-962d-53f1a14f05b6 OK (20.3s, 2 hablantes)
[14/50] 336a8602-1b56-4ca6-9dcc-7491b9a92a8a OK (12.7s, 2 hablantes)
[15/50

In [ ]:
import pandas as pd

df = pd.DataFrame(csv_rows)
df.to_csv(f"{DRIVE_OUT_DIR}/metadata.csv", index=False)
df.head(10)


,file_id,source,duration_sec,n_speakers_detected,n_segments
0,0445c357-e465-49ae-8067-bfa91d11b532,humano,309.52,2,106
1,09115a4a-ac50-4425-9dbc-b551a0b5342c,humano,87.20,2,36
2,0bc9a430-bdcd-44a7-a92d-e71dc33c8ad5,humano,105.64,1,14
3,0c6ada7d-d8c5-4840-ae79-0d8d573183dc,humano,141.48,1,12
4,0d2d37dd-38a4-40dd-ac62-b430f722ed57,humano,594.72,2,138
5,1c72dd55-acbb-4ebe-a4d7-446cc294a4a1,humano,224.62,1,57
6,1e3beb83-824b-4ad5-8483-f4ff6ea3c05e,humano,160.76,1,41
7,1f5cda00-bbf3-4770-97dc-99760ddadc1b,humano,143.74,2,48
8,2a92a64c-6ea8-48a0-b47a-445a3c577199,humano,626.20,2,191
9,2b377b6a-4fa1-4910-8d68-9ee05f7c302a,humano,202.66,1,46


## Revision de que se halla subido bien a google drive

In [ ]:
print("Total procesados:", len(df))
print("Distribución de hablantes detectados:")
print(df.groupby(["source", "n_speakers_detected"]).size())
print("Audios con hablantes != 2 (revisar manualmente):")
print(df[df["n_speakers_detected"] != 2][["file_id", "source", "n_speakers_detected", "duration_sec"]])


Total procesados: 100
Distribución de hablantes detectados:
source  n_speakers_detected
humano  1                      16
        2                      33
        3                       1
ia      1                       1
        2                      46
        3                       3
dtype: int64
Audios con hablantes != 2 (revisar manualmente):
                                 file_id  source  n_speakers_detected  \
2   0bc9a430-bdcd-44a7-a92d-e71dc33c8ad5  humano                    1   
3   0c6ada7d-d8c5-4840-ae79-0d8d573183dc  humano                    1   
5   1c72dd55-acbb-4ebe-a4d7-446cc294a4a1  humano                    1   
6   1e3beb83-824b-4ad5-8483-f4ff6ea3c05e  humano                    1   
9   2b377b6a-4fa1-4910-8d68-9ee05f7c302a  humano                    1   
16  440cb585-d675-4ed9-afa4-934e2a0aae90  humano                    1   
17  50e0f632-d195-4476-b049-3720482d65d9  humano                    1   
18  635a7986-ddd2-47f3-a6de-82a217053afa  humano              